In [1]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
selected_cols = [
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "issue_d",
    "purpose",
    "addr_state",
    "dti",
    "delinq_2yrs",
    "earliest_cr_line",
    "fico_range_low",
    "fico_range_high",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "mort_acc",
    "target"
]

df = pd.read_csv(
    "../data/processed/df_model_raw.csv",
    usecols=selected_cols,
    low_memory=False
)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

print("\nTarget distribution:")
print(df["target"].value_counts())

df.head()

Dataset loaded successfully.
Shape: (1348099, 25)

Target distribution:
target
0    1078739
1     269360
Name: count, dtype: int64


,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,purpose,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,open_acc,pub_rec,revol_bal,revol_util,total_acc,mort_acc,target
0,3600.0,36 months,13.99,123.03,C,C4,10+ years,MORTGAGE,55000.0,Not Verified,Dec-2015,debt_consolidation,PA,5.91,0.0,Aug-2003,675.0,679.0,7.0,0.0,2765.0,29.7,13.0,1.0,0
1,24700.0,36 months,11.99,820.28,C,C1,10+ years,MORTGAGE,65000.0,Not Verified,Dec-2015,small_business,SD,16.06,1.0,Dec-1999,715.0,719.0,22.0,0.0,21470.0,19.2,38.0,4.0,0
2,20000.0,60 months,10.78,432.66,B,B4,10+ years,MORTGAGE,63000.0,Not Verified,Dec-2015,home_improvement,IL,10.78,0.0,Aug-2000,695.0,699.0,6.0,0.0,7869.0,56.2,18.0,5.0,0
3,10400.0,60 months,22.45,289.91,F,F1,3 years,MORTGAGE,104433.0,Source Verified,Dec-2015,major_purchase,PA,25.37,1.0,Jun-1998,695.0,699.0,12.0,0.0,21929.0,64.5,35.0,6.0,0
4,11950.0,36 months,13.44,405.18,C,C3,4 years,RENT,34000.0,Source Verified,Dec-2015,debt_consolidation,GA,10.20,0.0,Oct-1987,690.0,694.0,5.0,0.0,8822.0,68.4,6.0,0.0,0


In [4]:
# Reduce memory usage by converting large numeric types into smaller types

for col in df.select_dtypes(include=["float64"]).columns:
    df[col] = pd.to_numeric(df[col], downcast="float")

for col in df.select_dtypes(include=["int64"]).columns:
    df[col] = pd.to_numeric(df[col], downcast="integer")

memory_mb = df.memory_usage(deep=True).sum() / 1e6
print(f"Memory usage after downcast: {memory_mb:.2f} MB")

Memory usage after downcast: 944.35 MB


In [5]:
def data_quality_report(df):
    report = pd.DataFrame({
        "dtype": df.dtypes,
        "null_count": df.isnull().sum(),
        "null_pct": (df.isnull().mean() * 100).round(2),
        "unique_values": df.nunique(),
        "sample_value": [
            df[col].dropna().iloc[0] if df[col].notnull().any() else "ALL NULL"
            for col in df.columns
        ]
    })

    report = report.sort_values("null_pct", ascending=False)

    print("Dataset shape:", df.shape)
    print("Total missing values:", df.isnull().sum().sum())
    print("Columns with missing values:", (report["null_count"] > 0).sum())

    return report

quality_report = data_quality_report(df)
quality_report

Dataset shape: (1348099, 25)
Total missing values: 130000
Columns with missing values: 10


,dtype,null_count,null_pct,unique_values,sample_value
emp_length,object,78550,5.83,11,10+ years
mort_acc,float32,50030,3.71,39,1.0
revol_util,float32,897,0.07,1379,29.700001
dti,float32,374,0.03,7067,5.91
loan_amnt,float32,0,0.00,1560,3600.0
grade,object,0,0.00,7,C
term,object,0,0.00,2,36 months
installment,float32,0,0.00,83531,123.029999
int_rate,float32,0,0.00,672,13.99
annual_inc,float64,4,0.00,64462,55000.0


In [6]:
quality_report.to_excel("../report/data_quality_report.xlsx")

print("Data quality report saved successfully.")

Data quality report saved successfully.


### Data Quality Audit Summary

The selected modeling dataset contains 1,348,099 records and 25 columns. A data quality report was generated to inspect data types, missing values, missing percentage, unique values, and sample values for each column.

The dataset contains 130,000 total missing values across 10 columns. No selected column has more than 40% missing values, so no column was dropped due to missingness. The most notable missing values are in `emp_length` and `mort_acc`, which will be handled during missing value imputation.

In [7]:
df["term"] = df["term"].str.strip().str.replace(" months", "")
df["term"] = pd.to_numeric(df["term"], errors="coerce")

In [8]:
print(df["term"].value_counts())
print(df["term"].dtype)

term
36    1023206
60     324893
Name: count, dtype: int64
int64


In [9]:
df["emp_length"].value_counts(dropna=False)

emp_length
10+ years    442679
2 years      122100
< 1 year     108537
3 years      107868
1 year        88843
5 years       84326
4 years       80763
NaN           78550
6 years       62879
8 years       60811
7 years       59724
9 years       51019
Name: count, dtype: int64

In [10]:
emp_map = {
    "< 1 year": 0,
    "1 year": 1,
    "2 years": 2,
    "3 years": 3,
    "4 years": 4,
    "5 years": 5,
    "6 years": 6,
    "7 years": 7,
    "8 years": 8,
    "9 years": 9,
    "10+ years": 10
}

df["emp_length"] = df["emp_length"].map(emp_map)

In [11]:
print(df["emp_length"].value_counts(dropna=False).sort_index())
print(df["emp_length"].dtype)

emp_length
0.0     108537
1.0      88843
2.0     122100
3.0     107868
4.0      80763
5.0      84326
6.0      62879
7.0      59724
8.0      60811
9.0      51019
10.0    442679
NaN      78550
Name: count, dtype: int64
float64


In [12]:
df["issue_d"].head()

0    Dec-2015
1    Dec-2015
2    Dec-2015
3    Dec-2015
4    Dec-2015
Name: issue_d, dtype: object

In [13]:
df["issue_d_backup"] = df["issue_d"]
df["issue_d"] = pd.to_datetime(
    df["issue_d"],
    format="%b-%Y",
    errors="coerce"
)
print(df[["issue_d_backup", "issue_d"]].head())
print("Missing dates after conversion:", df["issue_d"].isnull().sum())

  issue_d_backup    issue_d
0       Dec-2015 2015-12-01
1       Dec-2015 2015-12-01
2       Dec-2015 2015-12-01
3       Dec-2015 2015-12-01
4       Dec-2015 2015-12-01
Missing dates after conversion: 0


In [14]:
df[["issue_d_backup", "issue_d"]].sample(10, random_state=42)

,issue_d_backup,issue_d
20474,Dec-2015,2015-12-01
148031,Aug-2015,2015-08-01
974275,Mar-2017,2017-03-01
1224757,Aug-2016,2016-08-01
982265,Feb-2017,2017-02-01
927919,Aug-2011,2011-08-01
1096279,Jun-2013,2013-06-01
1227497,Aug-2016,2016-08-01
1066985,Aug-2013,2013-08-01
565754,Mar-2016,2016-03-01


In [15]:
print("Earliest issue date:", df["issue_d"].min())
print("Latest issue date:", df["issue_d"].max())

Earliest issue date: 2007-06-01 00:00:00
Latest issue date: 2018-12-01 00:00:00


In [16]:
df["issue_d"].dt.year.value_counts().sort_index()

issue_d
2007       603
2008      2393
2009      5281
2010     12537
2011     21721
2012     53367
2013    134804
2014    223103
2015    375546
2016    293105
2017    169321
2018     56318
Name: count, dtype: int64

In [17]:
print("Missing dates after conversion:", df["issue_d"].isnull().sum())

Missing dates after conversion: 0


In [18]:
df["earliest_cr_line"].head()

0    Aug-2003
1    Dec-1999
2    Aug-2000
3    Jun-1998
4    Oct-1987
Name: earliest_cr_line, dtype: object

In [19]:
df["earliest_cr_line"].sample(10, random_state=42)

20474      Dec-1997
148031     Apr-2007
974275     Mar-2010
1224757    Dec-1984
982265     Sep-2006
927919     Oct-2000
1096279    Apr-2000
1227497    Jan-2007
1066985    Apr-2002
565754     Mar-1999
Name: earliest_cr_line, dtype: object

In [20]:
df["earliest_cr_line_backup"] = df["earliest_cr_line"]
df["earliest_cr_line"] = pd.to_datetime(
    df["earliest_cr_line"],
    format="%b-%Y",
    errors="coerce"
)
print(df[["earliest_cr_line_backup", "earliest_cr_line"]].sample(10, random_state=42))

print("Earliest credit line:", df["earliest_cr_line"].min())
print("Latest credit line:", df["earliest_cr_line"].max())
print("Missing dates after conversion:", df["earliest_cr_line"].isnull().sum())
print("Data type:", df["earliest_cr_line"].dtype)

        earliest_cr_line_backup earliest_cr_line
20474                  Dec-1997       1997-12-01
148031                 Apr-2007       2007-04-01
974275                 Mar-2010       2010-03-01
1224757                Dec-1984       1984-12-01
982265                 Sep-2006       2006-09-01
927919                 Oct-2000       2000-10-01
1096279                Apr-2000       2000-04-01
1227497                Jan-2007       2007-01-01
1066985                Apr-2002       2002-04-01
565754                 Mar-1999       1999-03-01
Earliest credit line: 1934-04-01 00:00:00
Latest credit line: 2015-10-01 00:00:00
Missing dates after conversion: 29
Data type: datetime64[ns]


In [21]:
print(df["int_rate"].head())
print(df["int_rate"].dtype)

print(df["revol_util"].head())
print(df["revol_util"].dtype)

0    13.990000
1    11.990000
2    10.780000
3    22.450001
4    13.440000
Name: int_rate, dtype: float32
float32
0    29.700001
1    19.200001
2    56.200001
3    64.500000
4    68.400002
Name: revol_util, dtype: float32
float32


### Data Type Cleaning Summary

Several columns were converted into analysis-ready formats. The `term` column was cleaned from text values such as `36 months` and `60 months` into numeric values 36 and 60. The `emp_length` column was mapped from text categories such as `< 1 year`, `1 year`, and `10+ years` into numeric employment years from 0 to 10.

The date columns `issue_d` and `earliest_cr_line` were converted from month-year text format into datetime format. The columns `int_rate` and `revol_util` were checked and were already available as numeric columns, so no additional conversion was required.

In [22]:
df["emp_length"].median()

np.float64(6.0)

In [23]:
df["emp_length"] = df["emp_length"].fillna(df["emp_length"].median())
print("Missing emp_length:", df["emp_length"].isnull().sum())
print(df["emp_length"].value_counts().sort_index())

Missing emp_length: 0
emp_length
0.0     108537
1.0      88843
2.0     122100
3.0     107868
4.0      80763
5.0      84326
6.0     141429
7.0      59724
8.0      60811
9.0      51019
10.0    442679
Name: count, dtype: int64


In [24]:
df["mort_acc"].median()

np.float32(1.0)

In [25]:
df["mort_acc"] = df["mort_acc"].fillna(df["mort_acc"].median())

In [26]:
print("Missing mort_acc:", df["mort_acc"].isnull().sum())
print(df["mort_acc"].value_counts().sort_index().head(15))

Missing mort_acc: 0
mort_acc
0.0     523858
1.0     276174
2.0     188943
3.0     139385
4.0      94921
5.0      57633
6.0      32539
7.0      16852
8.0       8378
9.0       4285
10.0      2197
11.0      1206
12.0       640
13.0       356
14.0       244
Name: count, dtype: int64


In [27]:
df["revol_util"].median()

np.float32(52.2)

In [28]:
df["revol_util"] = df["revol_util"].fillna(df["revol_util"].median())

In [29]:
print("Missing revol_util:", df["revol_util"].isnull().sum())
print("Median revol_util:", df["revol_util"].median())

Missing revol_util: 0
Median revol_util: 52.2


In [30]:
df["dti"].median()

np.float32(17.61)

In [31]:
df["dti"] = df["dti"].fillna(df["dti"].median())
print("Missing dti:", df["dti"].isnull().sum())
print("Median dti:", df["dti"].median())

Missing dti: 0
Median dti: 17.61


In [32]:
df["annual_inc"].median()

np.float64(65000.0)

In [33]:
df["annual_inc"] = df["annual_inc"].fillna(df["annual_inc"].median())
print("Missing annual_inc:", df["annual_inc"].isnull().sum())
print("Median annual_inc:", df["annual_inc"].median())

Missing annual_inc: 0
Median annual_inc: 65000.0


In [34]:
for col in ["delinq_2yrs", "open_acc", "pub_rec", "total_acc"]:
    print(col, "median:", df[col].median(), "| missing:", df[col].isnull().sum())

delinq_2yrs median: 0.0 | missing: 29
open_acc median: 11.0 | missing: 29
pub_rec median: 0.0 | missing: 29
total_acc median: 23.0 | missing: 29


In [35]:
small_missing_cols = ["delinq_2yrs", "open_acc", "pub_rec", "total_acc"]

for col in small_missing_cols:
    df[col] = df[col].fillna(df[col].median())

print("Missing values after filling:")
print(df[small_missing_cols].isnull().sum())

Missing values after filling:
delinq_2yrs    0
open_acc       0
pub_rec        0
total_acc      0
dtype: int64


In [36]:
print("Rows before dropping:", df.shape[0])

df = df.dropna(subset=["earliest_cr_line"])

print("Rows after dropping:", df.shape[0])
print("Missing earliest_cr_line:", df["earliest_cr_line"].isnull().sum())

Rows before dropping: 1348099
Rows after dropping: 1348070
Missing earliest_cr_line: 0


In [37]:
print("Total missing values:", df.isnull().sum().sum())

missing_summary = df.isnull().sum()
missing_summary[missing_summary > 0]

Total missing values: 0


Series([], dtype: int64)

### Missing Value Treatment Summary

Missing values were handled based on column type and business meaning. Numeric columns such as `emp_length`, `mort_acc`, `revol_util`, `dti`, `annual_inc`, `delinq_2yrs`, `open_acc`, `pub_rec`, and `total_acc` were filled using median values because median is robust to outliers.

The `earliest_cr_line` column had only 29 missing values, so those rows were dropped because this column is required to calculate credit history length. After treatment, the dataset contains no missing values.

In [38]:
outlier_cols = ["annual_inc", "loan_amnt", "dti", "revol_bal", "open_acc", "total_acc"]

df[outlier_cols].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T

,count,mean,std,min,1%,25%,50%,75%,99%,max
annual_inc,1348070.0,76237.836628,69922.833774,0.0,18000.00,45750.86,65000.000000,90000.000000,251000.000000,10999200.0
loan_amnt,1348070.0,14409.225586,8716.088867,500.0,1500.00,7975.00,12000.000000,20000.000000,35000.000000,40000.0
dti,1348070.0,18.274290,11.153946,-1.0,1.77,11.79,17.610001,24.049999,38.459999,999.0
revol_bal,1348070.0,16271.058594,22477.623047,0.0,168.00,5937.00,11130.000000,19758.000000,95374.550000,2904836.0
open_acc,1348070.0,11.590447,5.474680,0.0,3.00,8.00,11.000000,14.000000,29.000000,90.0
total_acc,1348070.0,24.975990,12.003560,1.0,5.00,16.00,23.000000,32.000000,61.000000,176.0


In [39]:
col = "annual_inc"

lower_cap = df[col].quantile(0.01)
upper_cap = df[col].quantile(0.99)

print("Column:", col)
print("Before min:", df[col].min())
print("Before max:", df[col].max())
print("1% cap:", lower_cap)
print("99% cap:", upper_cap)

Column: annual_inc
Before min: 0.0
Before max: 10999200.0
1% cap: 18000.0
99% cap: 251000.0


In [40]:
df["annual_inc"] = df["annual_inc"].clip(lower=18000.0, upper=251000.0)
print("After capping annual_inc:")
print("Min:", df["annual_inc"].min())
print("Max:", df["annual_inc"].max())
print("1%:", df["annual_inc"].quantile(0.01))
print("99%:", df["annual_inc"].quantile(0.99))

After capping annual_inc:
Min: 18000.0
Max: 251000.0
1%: 18000.0
99%: 251000.0


In [41]:
col = "loan_amnt"

lower_cap = df[col].quantile(0.01)
upper_cap = df[col].quantile(0.99)

print("Column:", col)
print("Before min:", df[col].min())
print("Before max:", df[col].max())
print("1% cap:", lower_cap)
print("99% cap:", upper_cap)

Column: loan_amnt
Before min: 500.0
Before max: 40000.0
1% cap: 1500.0
99% cap: 35000.0


In [42]:
df["loan_amnt"] = df["loan_amnt"].clip(lower=1500.0, upper=35000.0)
print("After capping loan_amnt:")
print("Min:", df["loan_amnt"].min())
print("Max:", df["loan_amnt"].max())
print("1%:", df["loan_amnt"].quantile(0.01))
print("99%:", df["loan_amnt"].quantile(0.99))

After capping loan_amnt:
Min: 1500.0
Max: 35000.0
1%: 1500.0
99%: 35000.0


In [43]:
col = "dti"

lower_cap = df[col].quantile(0.01)
upper_cap = df[col].quantile(0.99)

print("Column:", col)
print("Before min:", df[col].min())
print("Before max:", df[col].max())
print("1% cap:", lower_cap)
print("99% cap:", upper_cap)

Column: dti
Before min: -1.0
Before max: 999.0
1% cap: 1.7699999809265137
99% cap: 38.459999084472656


In [44]:
df["dti"] = df["dti"].clip(
    lower=df["dti"].quantile(0.01),
    upper=df["dti"].quantile(0.99)
)
print("After capping dti:")
print("Min:", df["dti"].min())
print("Max:", df["dti"].max())
print("1%:", df["dti"].quantile(0.01))
print("99%:", df["dti"].quantile(0.99))

After capping dti:
Min: 1.77
Max: 38.46
1%: 1.7699999809265137
99%: 38.459999084472656


In [45]:
col = "revol_bal"

lower_cap = df[col].quantile(0.01)
upper_cap = df[col].quantile(0.99)

print("Column:", col)
print("Before min:", df[col].min())
print("Before max:", df[col].max())
print("1% cap:", lower_cap)
print("99% cap:", upper_cap)

Column: revol_bal
Before min: 0.0
Before max: 2904836.0
1% cap: 168.0
99% cap: 95374.55000000028


In [46]:
df["revol_bal"] = df["revol_bal"].clip(
    lower=df["revol_bal"].quantile(0.01),
    upper=df["revol_bal"].quantile(0.99)
)
print("After capping revol_bal:")
print("Min:", df["revol_bal"].min())
print("Max:", df["revol_bal"].max())
print("1%:", df["revol_bal"].quantile(0.01))
print("99%:", df["revol_bal"].quantile(0.99))

After capping revol_bal:
Min: 168.0
Max: 95374.55000000028
1%: 168.0
99%: 95373.48050000018


In [47]:
col = "open_acc"

lower_cap = df[col].quantile(0.01)
upper_cap = df[col].quantile(0.99)

print("Column:", col)
print("Before min:", df[col].min())
print("Before max:", df[col].max())
print("1% cap:", lower_cap)
print("99% cap:", upper_cap)

Column: open_acc
Before min: 0.0
Before max: 90.0
1% cap: 3.0
99% cap: 29.0


In [48]:
df["open_acc"] = df["open_acc"].clip(
    lower=df["open_acc"].quantile(0.01),
    upper=df["open_acc"].quantile(0.99)
)
print("After capping open_acc:")
print("Min:", df["open_acc"].min())
print("Max:", df["open_acc"].max())
print("1%:", df["open_acc"].quantile(0.01))
print("99%:", df["open_acc"].quantile(0.99))

After capping open_acc:
Min: 3.0
Max: 29.0
1%: 3.0
99%: 29.0


In [49]:
col = "total_acc"

lower_cap = df[col].quantile(0.01)
upper_cap = df[col].quantile(0.99)

print("Column:", col)
print("Before min:", df[col].min())
print("Before max:", df[col].max())
print("1% cap:", lower_cap)
print("99% cap:", upper_cap)

Column: total_acc
Before min: 1.0
Before max: 176.0
1% cap: 5.0
99% cap: 61.0


In [50]:
df["total_acc"] = df["total_acc"].clip(
    lower=df["total_acc"].quantile(0.01),
    upper=df["total_acc"].quantile(0.99)
)
print("After capping total_acc:")
print("Min:", df["total_acc"].min())
print("Max:", df["total_acc"].max())
print("1%:", df["total_acc"].quantile(0.01))
print("99%:", df["total_acc"].quantile(0.99))

After capping total_acc:
Min: 5.0
Max: 61.0
1%: 5.0
99%: 61.0


In [51]:
outlier_cols = ["annual_inc", "loan_amnt", "dti", "revol_bal", "open_acc", "total_acc"]

df[outlier_cols].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T

,count,mean,std,min,1%,25%,50%,75%,99%,max
annual_inc,1348070.0,74612.527560,42165.865097,18000.00,18000.00,45750.86,65000.000000,90000.000000,251000.000000,251000.000000
loan_amnt,1348070.0,14383.856445,8636.894531,1500.00,1500.00,7975.00,12000.000000,20000.000000,35000.000000,35000.000000
dti,1348070.0,18.127918,8.427738,1.77,1.77,11.79,17.610001,24.049999,38.459999,38.459999
revol_bal,1348070.0,15554.459130,15341.090522,168.00,168.00,5937.00,11130.000000,19758.000000,95373.480500,95374.550000
open_acc,1348070.0,11.546814,5.264616,3.00,3.00,8.00,11.000000,14.000000,29.000000,29.000000
total_acc,1348070.0,24.905918,11.698042,5.00,5.00,16.00,23.000000,32.000000,61.000000,61.000000


### Outlier Treatment Summary

Outliers were handled using percentile-based capping. For selected numeric columns, values below the 1st percentile were capped at the 1st percentile, and values above the 99th percentile were capped at the 99th percentile.

The following columns were capped:
- `annual_inc`
- `loan_amnt`
- `dti`
- `revol_bal`
- `open_acc`
- `total_acc`

This method reduces the impact of extreme values while preserving all rows in the dataset.

In [52]:
df["fico_avg"] = (df["fico_range_low"] + df["fico_range_high"]) / 2

print(df[["fico_range_low", "fico_range_high", "fico_avg"]].head())
print("Missing fico_avg:", df["fico_avg"].isnull().sum())

   fico_range_low  fico_range_high  fico_avg
0           675.0            679.0     677.0
1           715.0            719.0     717.0
2           695.0            699.0     697.0
3           695.0            699.0     697.0
4           690.0            694.0     692.0
Missing fico_avg: 0


In [53]:
df["credit_history_years"] = (
    (df["issue_d"] - df["earliest_cr_line"]).dt.days / 365.25
)

print(df[["issue_d", "earliest_cr_line", "credit_history_years"]].head())
print("Missing credit_history_years:", df["credit_history_years"].isnull().sum())
print("Minimum credit history:", df["credit_history_years"].min())
print("Maximum credit history:", df["credit_history_years"].max())

     issue_d earliest_cr_line  credit_history_years
0 2015-12-01       2003-08-01             12.334018
1 2015-12-01       1999-12-01             16.000000
2 2015-12-01       2000-08-01             15.331964
3 2015-12-01       1998-06-01             17.500342
4 2015-12-01       1987-10-01             28.167009
Missing credit_history_years: 0
Minimum credit history: 0.5037645448323066
Maximum credit history: 83.24982888432581


In [54]:
df["loan_to_income_ratio"] = df["loan_amnt"] / df["annual_inc"]

print(df[["loan_amnt", "annual_inc", "loan_to_income_ratio"]].head())
print("Missing loan_to_income_ratio:", df["loan_to_income_ratio"].isnull().sum())
print("Minimum:", df["loan_to_income_ratio"].min())
print("Maximum:", df["loan_to_income_ratio"].max())

   loan_amnt  annual_inc  loan_to_income_ratio
0     3600.0     55000.0              0.065455
1    24700.0     65000.0              0.380000
2    20000.0     63000.0              0.317460
3    10400.0    104433.0              0.099585
4    11950.0     34000.0              0.351471
Missing loan_to_income_ratio: 0
Minimum: 0.00597609561752988
Maximum: 1.9444444444444444


In [55]:
df["installment_to_income_ratio"] = df["installment"] / (df["annual_inc"] / 12)

print(df[["installment", "annual_inc", "installment_to_income_ratio"]].head())
print("Missing installment_to_income_ratio:", df["installment_to_income_ratio"].isnull().sum())
print("Minimum:", df["installment_to_income_ratio"].min())
print("Maximum:", df["installment_to_income_ratio"].max())

   installment  annual_inc  installment_to_income_ratio
0   123.029999     55000.0                     0.026843
1   820.280029     65000.0                     0.151436
2   432.660004     63000.0                     0.082411
3   289.910004    104433.0                     0.033312
4   405.179993     34000.0                     0.143005
Missing installment_to_income_ratio: 0
Minimum: 0.0005915999794006348
Maximum: 1.013280029296875


In [56]:
df["credit_history_years"] = df["credit_history_years"].round(2)

print(df[["issue_d", "earliest_cr_line", "credit_history_years"]].head())

     issue_d earliest_cr_line  credit_history_years
0 2015-12-01       2003-08-01                 12.33
1 2015-12-01       1999-12-01                 16.00
2 2015-12-01       2000-08-01                 15.33
3 2015-12-01       1998-06-01                 17.50
4 2015-12-01       1987-10-01                 28.17


In [57]:
df["issue_year"] = df["issue_d"].dt.year
df["issue_month"] = df["issue_d"].dt.month

print(df[["issue_d", "issue_year", "issue_month"]].head())
print("Missing issue_year:", df["issue_year"].isnull().sum())
print("Missing issue_month:", df["issue_month"].isnull().sum())

     issue_d  issue_year  issue_month
0 2015-12-01        2015           12
1 2015-12-01        2015           12
2 2015-12-01        2015           12
3 2015-12-01        2015           12
4 2015-12-01        2015           12
Missing issue_year: 0
Missing issue_month: 0


In [58]:
df["term_years"] = df["term"] / 12

print(df[["term", "term_years"]].head())
print("Missing term_years:", df["term_years"].isnull().sum())
print(df["term_years"].value_counts())

   term  term_years
0    36         3.0
1    36         3.0
2    60         5.0
3    60         5.0
4    36         3.0
Missing term_years: 0
term_years
3.0    1023177
5.0     324893
Name: count, dtype: int64


### Feature Engineering Summary

Several new features were created to improve credit risk analysis:

- `fico_avg`: Average of lower and upper FICO score range.
- `credit_history_years`: Number of years between loan issue date and earliest credit line.
- `loan_to_income_ratio`: Loan amount divided by annual income.
- `installment_to_income_ratio`: Monthly installment divided by estimated monthly income.
- `issue_year`: Year in which the loan was issued.
- `issue_month`: Month in which the loan was issued.
- `term_years`: Loan term converted from months to years.

These features help capture borrower creditworthiness, repayment burden, loan duration, and time-based lending patterns.

In [60]:
new_features = [
    "fico_avg",
    "credit_history_years",
    "loan_to_income_ratio",
    "installment_to_income_ratio",
    "issue_year",
    "issue_month",
    "term_years"
]

df[new_features].head()

,fico_avg,credit_history_years,loan_to_income_ratio,installment_to_income_ratio,issue_year,issue_month,term_years
0,677.0,12.33,0.065455,0.026843,2015,12,3.0
1,717.0,16.00,0.380000,0.151436,2015,12,3.0
2,697.0,15.33,0.317460,0.082411,2015,12,5.0
3,697.0,17.50,0.099585,0.033312,2015,12,5.0
4,692.0,28.17,0.351471,0.143005,2015,12,3.0


In [61]:
df[new_features].isnull().sum()

fico_avg                       0
credit_history_years           0
loan_to_income_ratio           0
installment_to_income_ratio    0
issue_year                     0
issue_month                    0
term_years                     0
dtype: int64

In [62]:
cat_cols = df.select_dtypes(include="object").columns

print("Categorical columns:")
print(cat_cols)

for col in cat_cols:
    print("\n", col)
    print(df[col].value_counts().head(10))

Categorical columns:
Index(['grade', 'sub_grade', 'home_ownership', 'verification_status', 'purpose', 'addr_state', 'issue_d_backup', 'earliest_cr_line_backup'], dtype='object')

 grade
grade
B    393092
C    382317
A    235182
D    201656
E     94191
F     32306
G      9326
Name: count, dtype: int64

 sub_grade
sub_grade
C1    85616
B4    83273
B5    82637
B3    81898
C2    79358
C3    75128
C4    74551
B2    74078
B1    71206
C5    67664
Name: count, dtype: int64

 home_ownership
home_ownership
MORTGAGE    666845
RENT        535682
OWN         145026
ANY            286
OTHER          182
NONE            49
Name: count, dtype: int64

 verification_status
verification_status
Source Verified    521579
Verified           418979
Not Verified       407512
Name: count, dtype: int64

 purpose
purpose
debt_consolidation    781441
credit_card           295625
home_improvement       87721
other                  78273
major_purchase         29550
medical                15614
small_business      

In [63]:
df = df.drop(columns=["issue_d_backup", "earliest_cr_line_backup"])

print(df.shape)
print(df.select_dtypes(include="object").columns)

(1348070, 32)
Index(['grade', 'sub_grade', 'home_ownership', 'verification_status', 'purpose', 'addr_state'], dtype='object')


In [64]:
grade_map = {
    "A": 1,
    "B": 2,
    "C": 3,
    "D": 4,
    "E": 5,
    "F": 6,
    "G": 7
}

df["grade_encoded"] = df["grade"].map(grade_map)

print(df[["grade", "grade_encoded"]].head(10))
print("Missing grade_encoded:", df["grade_encoded"].isnull().sum())
print(df["grade_encoded"].value_counts().sort_index())

  grade  grade_encoded
0     C              3
1     C              3
2     B              2
3     F              6
4     C              3
5     B              2
6     B              2
7     A              1
8     B              2
9     C              3
Missing grade_encoded: 0
grade_encoded
1    235182
2    393092
3    382317
4    201656
5     94191
6     32306
7      9326
Name: count, dtype: int64


In [65]:
sub_grade_order = {}

counter = 1
for grade in ["A", "B", "C", "D", "E", "F", "G"]:
    for num in ["1", "2", "3", "4", "5"]:
        sub_grade_order[grade + num] = counter
        counter += 1

df["sub_grade_encoded"] = df["sub_grade"].map(sub_grade_order)

print(df[["sub_grade", "sub_grade_encoded"]].head(10))
print("Missing sub_grade_encoded:", df["sub_grade_encoded"].isnull().sum())
print(df["sub_grade_encoded"].value_counts().sort_index().head())
print(df["sub_grade_encoded"].value_counts().sort_index().tail())

  sub_grade  sub_grade_encoded
0        C4                 14
1        C1                 11
2        B4                  9
3        F1                 26
4        C3                 13
5        B2                  7
6        B1                  6
7        A2                  2
8        B5                 10
9        C2                 12
Missing sub_grade_encoded: 0
sub_grade_encoded
1    43681
2    37187
3    38006
4    52254
5    64054
Name: count, dtype: int64
sub_grade_encoded
31    3033
32    2160
33    1644
34    1323
35    1166
Name: count, dtype: int64


In [66]:
verification_map = {
    "Not Verified": 0,
    "Source Verified": 1,
    "Verified": 2
}

df["verification_status_encoded"] = df["verification_status"].map(verification_map)

print(df[["verification_status", "verification_status_encoded"]].head(10))
print("Missing verification_status_encoded:", df["verification_status_encoded"].isnull().sum())
print(df["verification_status_encoded"].value_counts().sort_index())

  verification_status  verification_status_encoded
0        Not Verified                            0
1        Not Verified                            0
2        Not Verified                            0
3     Source Verified                            1
4     Source Verified                            1
5        Not Verified                            0
6        Not Verified                            0
7        Not Verified                            0
8        Not Verified                            0
9        Not Verified                            0
Missing verification_status_encoded: 0
verification_status_encoded
0    407512
1    521579
2    418979
Name: count, dtype: int64


In [67]:
home_dummies = pd.get_dummies(
    df["home_ownership"],
    prefix="home",
    drop_first=True
)

df = pd.concat([df, home_dummies], axis=1)

print(home_dummies.head())
print("New shape:", df.shape)

   home_MORTGAGE  home_NONE  home_OTHER  home_OWN  home_RENT
0           True      False       False     False      False
1           True      False       False     False      False
2           True      False       False     False      False
3           True      False       False     False      False
4          False      False       False     False       True
New shape: (1348070, 40)


In [68]:
home_dummy_cols = home_dummies.columns

df[home_dummy_cols] = df[home_dummy_cols].astype(int)

print(df[home_dummy_cols].head())

   home_MORTGAGE  home_NONE  home_OTHER  home_OWN  home_RENT
0              1          0           0         0          0
1              1          0           0         0          0
2              1          0           0         0          0
3              1          0           0         0          0
4              0          0           0         0          1


In [69]:
purpose_dummies = pd.get_dummies(
    df["purpose"],
    prefix="purpose",
    drop_first=True
)

df = pd.concat([df, purpose_dummies], axis=1)

purpose_dummy_cols = purpose_dummies.columns
df[purpose_dummy_cols] = df[purpose_dummy_cols].astype(int)

print(df[purpose_dummy_cols].head())
print("New shape:", df.shape)

   purpose_credit_card  purpose_debt_consolidation  purpose_educational  purpose_home_improvement  purpose_house  purpose_major_purchase  purpose_medical  purpose_moving  purpose_other  purpose_renewable_energy  purpose_small_business  purpose_vacation  purpose_wedding
0                    0                           1                    0                         0              0                       0                0               0              0                         0                       0                 0                0
1                    0                           0                    0                         0              0                       0                0               0              0                         0                       1                 0                0
2                    0                           0                    0                         1              0                       0                0               0              0      

In [70]:
state_dummies = pd.get_dummies(
    df["addr_state"],
    prefix="state",
    drop_first=True
)

df = pd.concat([df, state_dummies], axis=1)

state_dummy_cols = state_dummies.columns
df[state_dummy_cols] = df[state_dummy_cols].astype(int)

print(df[state_dummy_cols].head())
print("New shape:", df.shape)

   state_AL  state_AR  state_AZ  state_CA  state_CO  state_CT  state_DC  state_DE  state_FL  state_GA  state_HI  state_IA  state_ID  state_IL  state_IN  state_KS  state_KY  state_LA  state_MA  state_MD  state_ME  state_MI  state_MN  state_MO  state_MS  state_MT  state_NC  state_ND  state_NE  state_NH  state_NJ  state_NM  state_NV  state_NY  state_OH  state_OK  state_OR  state_PA  state_RI  state_SC  state_SD  state_TN  state_TX  state_UT  state_VA  state_VT  state_WA  state_WI  state_WV  state_WY
0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         0         1         0         0         0         0         0         0         0         0         0         0         0       

In [71]:
df.select_dtypes(include="object").columns

Index(['grade', 'sub_grade', 'home_ownership', 'verification_status', 'purpose', 'addr_state'], dtype='object')

In [72]:
drop_cols = [
    "grade",
    "sub_grade",
    "home_ownership",
    "verification_status",
    "purpose",
    "addr_state",
    "issue_d",
    "earliest_cr_line",
    "fico_range_low",
    "fico_range_high",
    "term"
]

df_clean = df.drop(columns=drop_cols)

print("Original df shape:", df.shape)
print("Clean modeling df shape:", df_clean.shape)

print("Remaining object columns:")
print(df_clean.select_dtypes(include="object").columns)

Original df shape: (1348070, 103)
Clean modeling df shape: (1348070, 92)
Remaining object columns:
Index([], dtype='object')


In [73]:
print("Shape:", df_clean.shape)

print("\nTotal missing values:")
print(df_clean.isnull().sum().sum())

print("\nObject columns:")
print(df_clean.select_dtypes(include="object").columns)

print("\nTarget distribution:")
print(df_clean["target"].value_counts())
print(df_clean["target"].value_counts(normalize=True) * 100)

Shape: (1348070, 92)

Total missing values:
0

Object columns:
Index([], dtype='object')

Target distribution:
target
0    1078713
1     269357
Name: count, dtype: int64
target
0    80.019064
1    19.980936
Name: proportion, dtype: float64


In [74]:
df_clean.to_csv("../data/processed/credit_risk_cleaned.csv", index=False)

print("Cleaned dataset saved successfully.")
print("Saved shape:", df_clean.shape)

Cleaned dataset saved successfully.
Saved shape: (1348070, 92)


## Phase 2 Summary: Data Cleaning and Feature Engineering

In this phase, the selected modeling dataset was cleaned and transformed for machine learning.

### Steps completed:

1. **Selected Column Loading**
   - Loaded only important borrower, loan, credit, and target columns to reduce memory usage.

2. **Data Type Cleaning**
   - Converted `term` from text to numeric months.
   - Converted `emp_length` into numeric years.
   - Converted `issue_d` and `earliest_cr_line` into datetime format.

3. **Missing Value Treatment**
   - Filled numeric missing values using median imputation.
   - Dropped 29 rows where `earliest_cr_line` was missing because it was required for credit history calculation.
   - Final dataset has 0 missing values.

4. **Outlier Treatment**
   - Applied 1st and 99th percentile capping to selected numeric columns:
     - `annual_inc`
     - `loan_amnt`
     - `dti`
     - `revol_bal`
     - `open_acc`
     - `total_acc`

5. **Feature Engineering**
   - Created `fico_avg`
   - Created `credit_history_years`
   - Created `loan_to_income_ratio`
   - Created `installment_to_income_ratio`
   - Created `issue_year`
   - Created `issue_month`
   - Created `term_years`

6. **Categorical Encoding**
   - Used ordinal encoding for ordered features:
     - `grade`
     - `sub_grade`
     - `verification_status`
   - Used one-hot encoding for nominal features:
     - `home_ownership`
     - `purpose`
     - `addr_state`

### Final Output

The cleaned modeling dataset was saved as:

`data/processed/credit_risk_cleaned.csv`

Final shape:

`(1348070, 92)`

The dataset has:
- No missing values
- No object columns
- Encoded categorical variables
- Engineered financial and credit-risk features
- Target distribution preserved at approximately 80% good loans and 20% bad loans